In [1]:
print("kernel test")

kernel test


In [2]:
file_path = "../gtfs_data/stops.txt"

In [3]:
import pandas as pd

In [4]:
import pandas as pd
df = pd.read_csv("../gtfs_data/stops.txt", low_memory=False)

In [5]:
df.columns.sort_values()

Index(['level_id', 'location_type', 'parent_station', 'platform_code',
       'stop_code', 'stop_id', 'stop_lat', 'stop_lon', 'stop_name',
       'wheelchair_boarding'],
      dtype='str')

In [12]:
df.count()

stop_id                170110
stop_code               89910
stop_name              170110
stop_lat               170110
stop_lon               170110
location_type           79812
parent_station          92330
wheelchair_boarding    168432
level_id                68138
platform_code             889
dtype: int64

In [ ]:
import pandas as pd
import folium

GTFS_DIR = "gtfs_data/"  # adjust to your path

# Load stops (you already have this as df)
stops = df.copy()
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")

# Get train route_ids from routes.txt (route_type 2 = rail)
routes = pd.read_csv("../gtfs_data/routes.txt", dtype=str)
train_routes = routes[routes["route_type"] == "2"]

# Get all trip shape_ids for train routes
trips = pd.read_csv("../gtfs_data/trips.txt", dtype=str)
train_trips = trips[trips["route_id"].isin(train_routes["route_id"])]
train_trip_ids = train_trips["trip_id"].unique()

# Get stop_ids from those trips (load only needed columns)
stop_times = pd.read_csv("../gtfs_data/stop_times.txt",dtype=str,usecols=["trip_id", "stop_id"])
train_stop_ids = stop_times[
    stop_times["trip_id"].isin(train_trip_ids)
]["stop_id"].unique()

# Filter stops to only train stops
train_stops = stops[stops["stop_id"].isin(train_stop_ids)].dropna(
    subset=["stop_lat", "stop_lon"]
)

print(f"Found {len(train_stops)} train stops")

# Build the map centred on Sydney
m = folium.Map(location=[-33.87, 151.21], zoom_start=11)

# Plot each stop
for _, stop in train_stops.iterrows():
    folium.CircleMarker(
        location=[stop["stop_lat"], stop["stop_lon"]],
        radius=5,
        color="#E74C3C",
        fill=True,
        fill_color="#E74C3C",
        fill_opacity=0.8,
        popup=folium.Popup(f"{stop['stop_name']}<br>{stop['stop_id']}", max_width=200)
    ).add_to(m)

m.save("train_stops.html")
print("Saved to train_stops.html")

Found 714 train stops
Saved to train_stops.html


In [18]:
import pandas as pd
import json

# Your existing df with all train stops
# Deduplicate by parent station to get one point per station (not per platform)
stations = (
    df[df["parent_station"].notna()]  # child stops (actual platforms)
    .dropna(subset=["stop_lat", "stop_lon"])
    .copy()
)

# Extract station name without platform suffix
stations["station_name"] = stations["stop_name"].str.replace(r",?\s*Platform.*$", "", regex=True).str.strip()

# One entry per unique station
unique_stations = (
    stations
    .groupby("station_name", as_index=False)
    .agg({"stop_lat": "first", "stop_lon": "first", "stop_id": "first"})
    .rename(columns={"stop_id": "id", "stop_lat": "lat", "stop_lon": "lon", "station_name": "name"})
    .sort_values("name")
)

# Export as JSON for the simulator
train_stops_json = unique_stations[["id", "name", "lat", "lon"]].to_dict(orient="records")
print(f"Found {len(unique_stations)} unique stations")
print(json.dumps(train_stops_json[:3], indent=2))  # preview

Found 87909 unique stations
[
  {
    "id": "225030_PCE2",
    "name": "",
    "lat": -33.44621517,
    "lon": 151.32838071
  },
  {
    "id": "2400140",
    "name": "1 Boggabilla Rd",
    "lat": -29.45763611,
    "lon": 149.84276981
  },
  {
    "id": "2536172",
    "name": "1 Country Club Dr",
    "lat": -35.72212888,
    "lon": 150.18860588
  }
]


In [19]:
with open("train_stops.json", "w") as f:
    json.dump(train_stops_json, f)
print("Saved train_stops.json")

Saved train_stops.json


In [1]:
print("hello")

hello


In [2]:
stops.columns.sort_values()

NameError: name 'stops' is not defined